# Week 43, Day 1: Place cells: rate maps, spatial information and remapping

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w043/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: From spikes to a rate map

Write ratemap(s): histogram the positions into 20 by 20 bins, weighted by the spike counts s, and divide by the occupancy (seconds per bin). Leave unvisited bins as NaN.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(1971)
dt = 0.02; T = int(20 * 60 / dt)                    # 20 minutes at 50 Hz
pos = np.zeros((T, 2)); p = np.array([0.5, 0.5]); hd = 0.0
for t in range(T):                                   # a random forager in a 1 m box
    hd += 0.3 * r.normal()
    q = p + 0.15 * dt * np.array([np.cos(hd), np.sin(hd)])
    if not (0 < q[0] < 1 and 0 < q[1] < 1):
        hd += np.pi; q = p
    p = q; pos[t] = p

N, sig, peak = 30, 0.08, 15.0
cent = r.uniform(0.1, 0.9, (N, 2))
lam = peak * np.exp(-((pos[:, None, :] - cent[None]) ** 2).sum(-1) / (2 * sig ** 2))
spikes = r.poisson(lam * dt)                         # T x N spike counts

nb = 20
occ, _, _ = np.histogram2d(pos[:, 0], pos[:, 1], bins=nb, range=[[0, 1], [0, 1]])
occ = occ * dt                                       # seconds in each bin

def ratemap(s):
    ...

g = (np.arange(nb) + 0.5) / nb; X, Y = np.meshgrid(g, g, indexing="ij")
true0 = peak * np.exp(-((X - cent[0, 0]) ** 2 + (Y - cent[0, 1]) ** 2) / (2 * sig ** 2))
m0 = ratemap(spikes[:, 0])
print(f"bins visited {(occ > 0).mean():.3f}; correlation with the true field {np.corrcoef(m0[occ > 0], true0[occ > 0])[0, 1]:.3f}")
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].plot(pos[:, 0], pos[:, 1], color="0.8", lw=0.3); s = spikes[:, 0] > 0; ax[0].plot(pos[s, 0], pos[s, 1], "r.", ms=2)
ax[1].imshow(m0.T, origin="lower", extent=[0, 1, 0, 1]); plt.show()

In [ ]:
import numpy as np
def test_cover():
    "the rat visits almost every bin"
    assert (occ > 0).mean() > 0.95

def test_map():
    "the estimated rate map correlates above 0.85 with the true field"
    ok = occ > 0
    assert np.corrcoef(m0[ok], true0[ok])[0, 1] > 0.85

test_cover()
test_map()
print("All checks passed")

### Exercise: Information, bias and the shuffle

Write skaggs(m) using the visited bins only, then shuffle_null(s, n=100), which returns the 99th percentile of the information over n random circular shifts of the spike train by at least 20 s.

In [ ]:
ok = occ > 0

def skaggs(m):
    p = occ[ok] / occ[ok].sum(); lam_i = m[ok]
    lam_bar = ...
    with np.errstate(divide="ignore", invalid="ignore"):
        terms = np.where(lam_i > 0, ..., 0.0)
    return terms.sum()

def shuffle_null(s, n=100):
    shift_min = int(20 / dt)
    vals = [skaggs(ratemap(np.roll(s, r.integers(shift_min, T - shift_min)))) for _ in range(n)]
    return np.percentile(vals, 99)

flat = r.poisson(np.full(T, 2.0) * dt)               # a cell that ignores position
I_place, I_flat = skaggs(m0), skaggs(ratemap(flat))
null_place, null_flat = shuffle_null(spikes[:, 0]), shuffle_null(flat)
print(f"place cell: {I_place:.2f} bits/spike (99% of shuffles below {null_place:.2f})")
print(f"flat cell:  {I_flat:.3f} bits/spike (99% of shuffles below {null_flat:.3f})")

In [ ]:
import numpy as np
def test_place():
    "the place cell carries more than 3 bits per spike and beats its shuffles"
    assert I_place > 3 and I_place > null_place
def test_bias():
    "the non-spatial cell has positive measured information but does not beat its shuffles"
    assert I_flat > 0 and I_flat < null_flat

test_place()
test_bias()
print("All checks passed")

### Exercise: Same room, new room

Write mapcorr(a, b): the Pearson correlation of two rate maps over bins visited in both. Use it to compare the first and second halves of the session, and room A with room B, where each cell's field has moved to a random new centre.

In [ ]:
half = T // 2
occ1, _, _ = np.histogram2d(pos[:half, 0], pos[:half, 1], bins=nb, range=[[0, 1], [0, 1]])
occ2, _, _ = np.histogram2d(pos[half:, 0], pos[half:, 1], bins=nb, range=[[0, 1], [0, 1]])

def map_from(s, idx, o):
    c, _, _ = np.histogram2d(pos[idx, 0], pos[idx, 1], bins=nb, range=[[0, 1], [0, 1]], weights=s[idx])
    with np.errstate(invalid="ignore", divide="ignore"):
        return c / (o * dt)

def mapcorr(a, b):
    ...

cent_B = r.uniform(0.1, 0.9, (N, 2))                 # room B: every field moves
lam_B = peak * np.exp(-((pos[:, None, :] - cent_B[None]) ** 2).sum(-1) / (2 * sig ** 2))
spikes_B = r.poisson(lam_B * dt)
first, second = np.arange(half), np.arange(half, T)
r_same = [mapcorr(map_from(spikes[:, i], first, occ1), map_from(spikes[:, i], second, occ2)) for i in range(N)]
r_new = [mapcorr(ratemap(spikes[:, i]), ratemap(spikes_B[:, i])) for i in range(N)]
print(f"same room, two halves: mean r = {np.mean(r_same):.2f};  room A against room B: mean r = {np.mean(r_new):.2f}")

In [ ]:
import numpy as np
def test_same():
    "maps from the two halves of a session correlate above 0.7 on average"
    assert np.mean(r_same) > 0.7
def test_new():
    "maps from different rooms correlate below 0.15 on average"
    assert abs(np.mean(r_new)) < 0.15

test_same()
test_new()
print("All checks passed")